# `indic/02` — Apply IndicXlit Romanisation

**Purpose:** Apply the IndicXlit neural transliteration system to the target side
of the IndicMT Eval corpus, converting native-script `Translation` and `Reference`
columns into Latin script.

The intervention is **script-only**: phonological, morphological, and semantic
content is preserved. `Source` (English) and `DA_score` are left untouched.

**Reads:** `data/processed/<language>_indicmt.csv` (produced by `indic/01`)

**Outputs produced (new columns added in-place):**
```
data/processed/gujarati_indicmt.csv   -- adds: Translation_Transliteration_romanized, Reference_Transliteration_romanized
data/processed/hindi_indicmt.csv      -- adds: Translation_Transliteration_romanized, Reference_Transliteration_romanized
data/processed/malayalam_indicmt.csv  -- adds: Translation_Transliteration_romanized, Reference_Transliteration_romanized
data/processed/marathi_indicmt.csv    -- adds: Translation_Transliteration_romanized, Reference_Transliteration_romanized
data/processed/tamil_indicmt.csv      -- adds: Translation_Transliteration_romanized, Reference_Transliteration_romanized
```

In [ ]:
import subprocess, sys
try:
    from ai4bharat.transliteration import XlitEngine
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ai4bharat-transliteration', '-q'])
    from ai4bharat.transliteration import XlitEngine
import pandas as pd
print('IndicXlit ready.')

## Configuration

All paths are relative to the `notebooks/indic/` directory.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../../data/processed")

# Keys = filename prefixes (must match nb01 output)
# Values = IndicXlit language codes
LANG_CONFIGS = {
    "gujarati":  "gu",
    "hindi":     "hi",
    "malayalam": "ml",
    "marathi":   "mr",
    "tamil":     "ta",
}

# Columns to romanise -> output column names (matches nb03-12 schema)
TARGET_COLS = {
    "Translation": "Translation_Transliteration_romanized",
    "Reference":   "Reference_Transliteration_romanized",
}

print(f"Data directory : {DATA_DIR.resolve()}")

## Step 1 — Load Per-Language CSVs

In [ ]:
raw = {}

for lang in LANG_CONFIGS:
    path = DATA_DIR / f"{lang}_indicmt.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run indic/01_fetch_indicmt_eval.ipynb first.")
    df = pd.read_csv(path)
    raw[lang] = df
    present = [c for c in TARGET_COLS if c in df.columns]
    missing = [c for c in TARGET_COLS if c not in df.columns]
    print(f"  {lang}  {len(df):,} rows  |  found: {present}  |  missing: {missing}")

print(f"\nLoaded {len(raw)} language files.")

## Step 2 — Load IndicXlit Engine

In [ ]:
engines = {}
for lang, lang_code in LANG_CONFIGS.items():
    engines[lang] = XlitEngine(lang_code, beam_width=4, rescore=False)
    print(f"  Loaded engine for {lang} ({lang_code})")

## Step 3 — Transliterate `Translation` and `Reference`

In [ ]:
import warnings

def translit_series(series, engine, lang_code):
    out = []
    for idx, text in series.items():
        if pd.isna(text) or str(text).strip() == '':
            out.append('')
            continue
        try:
            out.append(engine.translit_sentence(str(text), lang_code))
        except Exception as exc:
            warnings.warn(f"Row {idx}: {exc} -- keeping original")
            out.append(str(text))
    return pd.Series(out, index=series.index)

romanised = {}

for lang, df in raw.items():
    d = df.copy()
    eng = engines[lang]
    lc  = LANG_CONFIGS[lang]
    for src_col, tgt_col in TARGET_COLS.items():
        if src_col in d.columns:
            d[tgt_col] = translit_series(d[src_col], eng, lc)
        else:
            print(f"  WARNING: {lang} missing column '{src_col}' -- skipping")
    romanised[lang] = d
    print(f"  {lang}: romanisation done")

## Step 4 — Sanity Check

In [ ]:
for lang, df in romanised.items():
    if 'Translation' not in df.columns or 'Translation_Transliteration_romanized' not in df.columns:
        continue
    sample = df[['Translation', 'Translation_Transliteration_romanized']].dropna().sample(
        min(3, len(df)), random_state=42)
    print(f"{'='*55}")
    print(f"{lang} -- Translation (native) vs Translation_Transliteration_romanized")
    print(f"{'='*55}")
    for _, row in sample.iterrows():
        print(f"  native : {row['Translation']}")
        print(f"  roman  : {row['Translation_Transliteration_romanized']}")
        print()

## Step 5 — Save Updated CSVs

In [ ]:
for lang, df in romanised.items():
    out_path = DATA_DIR / f"{lang}_indicmt.csv"
    df.to_csv(out_path, index=False)
    new_cols = [c for c in df.columns if 'romanized' in c]
    print(f"  Saved  {out_path}  ({len(df):,} rows)  new cols: {new_cols}")

print(f"\n  {len(romanised)} files updated in {DATA_DIR.resolve()}")

## Step 6 — Romanisation Coverage Summary

In [ ]:
print(f"{'Lang':<12}  {'Rows':>6}  {'Translation_rom filled':>24}  {'Reference_rom filled':>22}")
print('-' * 70)
for lang, df in romanised.items():
    n = len(df)
    t_col = 'Translation_Transliteration_romanized'
    r_col = 'Reference_Transliteration_romanized'
    t_filled = df[t_col].str.strip().str.len().gt(0).sum() if t_col in df.columns else 0
    r_filled = df[r_col].str.strip().str.len().gt(0).sum() if r_col in df.columns else 0
    print(f'  {lang:<10}  {n:>5,}    {t_filled:>6,} / {n:<5,}    {r_filled:>6,} / {n:<5,}')
print('\n  Romanisation complete. data/processed/ is ready for indic/03.')